In [0]:
display(
    spark.sql("""
        SHOW TABLES IN workspace.gold
    """)
)

database,tableName,isTemporary
gold,dim_cliente,false
gold,dim_fecha,false
gold,dim_producto,false
gold,dim_sucursal,false
gold,dim_vendedor,false
gold,fact_metas,false
gold,fact_ventas,false


In [0]:
tablas_gold = [
    "dim_cliente",
    "dim_fecha",
    "dim_producto",
    "dim_sucursal",
    "dim_vendedor",
    "fact_ventas",
    "fact_metas"
]

for tabla in tablas_gold:
    cantidad = spark.table(f"workspace.gold.{tabla}").count()
    print(f"{tabla}: {cantidad} registros")

dim_cliente: 100 registros
dim_fecha: 577 registros
dim_producto: 50 registros
dim_sucursal: 10 registros
dim_vendedor: 20 registros
fact_ventas: 10000 registros
fact_metas: 380 registros


In [0]:
display(
    spark.sql("""
        SELECT
            SUM(CASE WHEN c.ClienteCodigo IS NULL THEN 1 ELSE 0 END)
                AS ClientesNoEncontrados,

            SUM(CASE WHEN p.ProductoCodigo IS NULL THEN 1 ELSE 0 END)
                AS ProductosNoEncontrados,

            SUM(CASE WHEN v.VendedorCodigo IS NULL THEN 1 ELSE 0 END)
                AS VendedoresNoEncontrados,

            SUM(CASE WHEN s.SucursalCodigo IS NULL THEN 1 ELSE 0 END)
                AS SucursalesNoEncontradas,

            SUM(CASE WHEN f.FechaID IS NULL THEN 1 ELSE 0 END)
                AS FechasNoEncontradas

        FROM workspace.gold.fact_ventas x

        LEFT JOIN workspace.gold.dim_cliente c
            ON x.ClienteCodigo = c.ClienteCodigo

        LEFT JOIN workspace.gold.dim_producto p
            ON x.ProductoCodigo = p.ProductoCodigo

        LEFT JOIN workspace.gold.dim_vendedor v
            ON x.VendedorCodigo = v.VendedorCodigo

        LEFT JOIN workspace.gold.dim_sucursal s
            ON x.SucursalCodigo = s.SucursalCodigo

        LEFT JOIN workspace.gold.dim_fecha f
            ON x.FechaID = f.FechaID
    """)
)

ClientesNoEncontrados,ProductosNoEncontrados,VendedoresNoEncontrados,SucursalesNoEncontradas,FechasNoEncontradas
0,0,0,0,0


In [0]:
display(
    spark.sql("""
        SELECT
            COUNT(*) AS TotalRegistros,
            SUM(
                CASE
                    WHEN VentaBruta <> Cantidad * PrecioUnitario
                    THEN 1 ELSE 0
                END
            ) AS ErrorVentaBruta,

            SUM(
                CASE
                    WHEN CostoTotal <> Cantidad * CostoUnitario
                    THEN 1 ELSE 0
                END
            ) AS ErrorCostoTotal,

            SUM(
                CASE
                    WHEN Margen <> VentaBruta - CostoTotal
                    THEN 1 ELSE 0
                END
            ) AS ErrorMargen

        FROM workspace.gold.fact_ventas
    """)
)

TotalRegistros,ErrorVentaBruta,ErrorCostoTotal,ErrorMargen
10000,0,0,0


In [0]:
display(
    spark.sql("""
        SELECT VentaCodigo, COUNT(*) AS Cantidad
        FROM workspace.gold.fact_ventas
        GROUP BY VentaCodigo
        HAVING COUNT(*) > 1
    """)
)

VentaCodigo,Cantidad


In [0]:
display(
    spark.sql("""
        SELECT MetaCodigo, COUNT(*) AS Cantidad
        FROM workspace.gold.fact_metas
        GROUP BY MetaCodigo
        HAVING COUNT(*) > 1
    """)
)

MetaCodigo,Cantidad
